In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv() # carrega a API da I.A do .env

model = init_chat_model("openai:gpt-4o-mini")

In [ ]:
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings

load_dotenv()
embeddings = init_embeddings("google_genai:gemini-embedding-001")

vetor = embeddings.embed_query("Como funciona a memória na v1?")
print(len(vetor))   # ex.: 1536 números // varia conforme cada LLM

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)      #indexa os chunks (gera e guarda os vetores)

# Busca direta por similaridade:
resultados = vector_store.similarity_search("Como funciona a memória na v1?", k=2)
for doc in resultados:
    print("->", doc.page_content[:60], "...")

# Com a PONTUAÇÃO de similaridade de cada resultado:
for doc, score in vector_store.similarity_search_with_score("memória v1", K=2):
    print(round(score, 3), doc.metadata.get("fonte"))

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})       # 2 trechos mais relevantes
docs = retriever.invoke("Como funciona a memória na v1?")           # -> list[Document]

In [ ]:
vector_store.dump("indice.json")                                # salva no disco
vs2 = InMemoryVectorStore.load("indice.json", embeddings)       # recarrega depois

In [ ]:
# Weaviate — banco de vetores que persiste no servidor (Docker, embedded ou cloud)
# pip install langchain-weaviate weaviate-client
import weaviate
from langchain_weaviate.vectorstores import WeaviateVectorStore

# Conecta a um Weaviate local (ex.: rodando em Docker, na porta 8080).
client = weaviate.connect_to_local()

vector_store = WeaviateVectorStore.from_documents(
    chunks,
    embedding=embeddings,        # usamos NOSSOS embeddings (init_embeddings)
    client=client,
    index_name="MeusDocs",       # nome da coleção (começa com letra maiúscula)
)
vector_store.similarity_search("memória v1", k=2)

client.close()                   # feche a conexão ao terminar

In [ ]:
# FAISS — índice salvo/recarregado de arquivos
from langchain_community.vectorstores import FAISS

vs_faiss = FAISS.from_documents(chunks, embeddings)
vs_faiss.save_local("faiss_index")
# allow_dangerous_deserialization=True: você confia no arquivo que criou
vs_faiss = FAISS.load_local("faiss_index", embeddings,
allow_dangerous_deserialization=True)

In [ ]:
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import init_embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

load_dotenv()  # OPENAI_API_KEY do .env (init_embeddings usa a OpenAI)

# 1. LOADER — carrega o conteúdo em Documents
# docs = TextLoader("dados/manual.txt", encoding="utf-8").load()

from langchain_core.documents import Document

textos = [
    "O leão é o rei da floresta.",
    "Elefantes Thailandeses são animais bonitos.",
    "Gatos adoram dormir em lugares quentinhos.",
    "Cachorros latem e correm ao redor da casa."
]

docs = [Document(page_content=texto) for texto in textos]

In [ ]:
# 2. SPLITTER — quebra em chunks com sobreposição
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
chunks = splitter.split_documents(docs)

# 3. EMBEDDINGS — modelo que transforma texto em vetor
embeddings = init_embeddings("google_genai:gemini-embedding-001")

# 4. VECTOR STORE — guarda os vetores e permite busca por similaridade
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

# 5. RETRIEVER — interface de busca pronta para chains/tools
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [ ]:
# Teste:
for doc in retriever.invoke("Animais para se ver num Zoológico ou safari"):
    print(doc.metadata.get("start_index"), "->", doc.page_content[:70], "...")